# Week 5 — Personal Protective Equipment (PPE) Detection: YOLO11n

Fine-tunes **YOLO11n** on the Roboflow Construction PPE dataset to detect:
- Safety helmets (`hardhat` / `no_hardhat`)
- High-visibility safety vests (`vest` / `no_vest`)
- Construction personnel (`person`)
- Associated gear (`boots`, `no_boots`, `gloves`, `no_gloves`)

Produces the trained model checkpoint at `models/ppe_yolo11n.pt` that `backend/pipeline/ppe.py`
loads for real-time compliance verification in the live video pipeline.

## 1. Setup & Environment Check

In [1]:
import sys
import os
import json
import time
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from ultralytics import YOLO

DATA_YAML = PROJECT_ROOT / "data" / "ppe" / "data_local.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "ppe_yolo11n.pt"
MANIFEST_PATH = PROJECT_ROOT / "models" / "ppe_yolo11n.manifest.json"

print("Project Root:   ", PROJECT_ROOT)
print("Dataset Config: ", DATA_YAML, "(Exists: ", DATA_YAML.exists(), ")")
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:     ", torch.cuda.get_device_name(0))
    print("GPU VRAM:       ", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Project Root:    c:\Users\athar\Downloads\Development of a Multimodal Vision-Language
Dataset Config:  c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\data\ppe\data_local.yaml (Exists:  True )
CUDA Available:  True
GPU Device:      NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU VRAM:        6.44 GB


## 2. Dataset Overview

The dataset is configured via `data/ppe/data_local.yaml` with the following classes:
0. `boots`
1. `gloves`
2. `hardhat` (Safety helmet)
3. `no_boots`
4. `no_gloves`
5. `no_hardhat` (Missing helmet)
6. `no_vest` (Missing safety vest)
7. `person`
8. `vest` (High-visibility vest)

In [2]:
# Inspect sample dataset image count
for split in ["train", "valid", "test"]:
    img_dir = PROJECT_ROOT / "data" / "ppe" / split / "images"
    if img_dir.exists():
        print(f"{split:6s}: {len(list(img_dir.glob('*.jpg')))} images")

train : 75 images
valid : 25 images
test  : 100 images


## 3. YOLO11n Fine-Tuning

Fine-tunes `yolo11n.pt` with transfer learning on the PPE dataset using the local GPU.

In [3]:
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolo11n.pt")

print("Starting training on device:", device)
results = model.train(
    data=str(DATA_YAML),
    epochs=25,
    batch=16,
    imgsz=640,
    device=device,
    workers=2,
    project=str(PROJECT_ROOT / "runs" / "ppe"),
    name="notebook_run",
    exist_ok=True,
    save=True,
)

Starting training on device: 0
New https://pypi.org/project/ultralytics/8.4.175 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.160  Python-3.14.2 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\data\ppe\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, ke

## 4. Evaluation & Metrics (mAP@0.5, Precision, Recall)

In [4]:
# Load best checkpoint and evaluate on validation split
best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
val_model = YOLO(str(best_pt))
metrics = val_model.val(data=str(DATA_YAML), device=device)

print("=" * 40)
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")
print("=" * 40)

Ultralytics 8.4.160  Python-3.14.2 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLO11n summary (fused): 100 layers, 2,583,907 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 523.6146.6 MB/s, size: 56.8 KB)
val: Scanning C:\Users\athar\Downloads\Development of a Multimodal Vision-Language\data\ppe\valid\labels.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 10.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.9s/it 3.9s<11.4s
                   all         25        405      0.584      0.278      0.332      0.159
                 boots         17         75      0.362    0.00966      0.207     0.0523
                gloves         13         40          1          0    0.00437   0.000994
               hardhat         25         86      0.831      0.733      0.819      0.421
             no_gloves         12         39     

## 5. Export Checkpoint & Write Reproducibility Manifest (NFR-12)

In [5]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, CHECKPOINT_PATH)
print(f"Copied best weights to: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / 1e6:.2f} MB)")

manifest = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "model": {
        "architecture": "YOLO11n-PPE",
        "base_checkpoint": "yolo11n.pt",
        "classes": val_model.names,
    },
    "dataset": {
        "source": "Roboflow PPE Dataset v3",
        "config_path": str(DATA_YAML),
    },
    "metrics": {
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    },
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(f"Reproducibility manifest written to: {MANIFEST_PATH}")

Copied best weights to: c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\models\ppe_yolo11n.pt (5.47 MB)
Reproducibility manifest written to: c:\Users\athar\Downloads\Development of a Multimodal Vision-Language\models\ppe_yolo11n.manifest.json
